# ENA raw data runs: test examples

Shows what `ena_runs.py` returns on a few typical cases.
Only ENA's search answer is read (a small text with the metadata of each run); no reads file (FASTQ) is downloaded.
ENA also contains the runs of SRA and DDBJ, so one request covers the three databases. GSA (China) is not included.

In [1]:
import sys
sys.path.append("../..")

import json
from collections import Counter
from raw_data.scripts_raw_data.ena_runs import ena_runs
from artifact_links.scripts_artifact_links.common import LinkError


def show(taxid, data_type="RNA-Seq", max_runs=5000):
    """Print how many runs ENA has and how many we got, or the problem if there is one."""
    try:
        result = ena_runs(taxid, data_type, max_runs)
    except LinkError as error:
        print("PROBLEM:", error)
        return None
    print("taxid:", result["taxid"], "| data type:", result["data_type"])
    print("runs in ENA:", result["total_runs"], "| runs returned:", len(result["runs"]), "| complete:", result["complete"])
    return result

## 1. All the RNA-seq runs of a species (cat)

The cat has 2,000 RNA-seq runs, fewer than the limit (5,000), so we get all of them (`complete` is `True`).
The first letters of a run accession show where it was submitted: SRR = SRA (NCBI), ERR = ENA, DRR = DDBJ.

In [2]:
cat = show(9685)
Counter(run["run_accession"][:3] for run in cat["runs"])

taxid: 9685 | data type: RNA-Seq
runs in ENA: 2000 | runs returned: 2000 | complete: True


Counter({'SRR': 1873, 'DRR': 69, 'ERR': 58})

## 2. One run, cleaned

A paired-end run (`layout` PAIRED) has two reads files, `_1` and `_2`. Each file has its link, its size in bytes and its md5.
ENA gives the links without `https://`, the script adds it. Empty fields become `null`.

In [3]:
paired = [run for run in cat["runs"] if run["layout"] == "PAIRED"]
print(json.dumps(paired[0], indent=2))

{
  "run_accession": "SRR12809844",
  "study_accession": "PRJNA668739",
  "sample_accession": "SAMN16419321",
  "study_title": "Felis catus host response to feline coronavirus infection RNASeq reads",
  "sample_title": "CRFK_Infec_T17_R3",
  "layout": "PAIRED",
  "library_source": "TRANSCRIPTOMIC",
  "instrument": "Illumina NovaSeq 6000",
  "read_count": 53566614,
  "base_count": 16069984200,
  "fastq": [
    {
      "url": "https://ftp.sra.ebi.ac.uk/vol1/fastq/SRR128/044/SRR12809844/SRR12809844_1.fastq.gz",
      "size": 3611676455,
      "md5": "c3d0491a94fd168cb02719be650f7a20"
    },
    {
      "url": "https://ftp.sra.ebi.ac.uk/vol1/fastq/SRR128/044/SRR12809844/SRR12809844_2.fastq.gz",
      "size": 3655352047,
      "md5": "f1033b418bb42ba7baea4da49871df27"
    }
  ],
  "tissue_type": "kidney",
  "cell_type": null,
  "dev_stage": null,
  "sex": "female"
}


## 3. Cleaned metadata

The submitters write the same value in different ways. The script puts tissue, cell type, dev stage and sex in lower case,
so "Left ventricle" and "left ventricle" are counted together. Words like "missing" or "unknown" become `None`.
Only `tissue_type` is filled for most runs; the other fields are often empty.

In [4]:
for field in ["tissue_type", "cell_type", "dev_stage", "sex"]:
    filled = [run[field] for run in cat["runs"] if run[field] is not None]
    print(field, len(filled), "/", len(cat["runs"]), Counter(filled).most_common(4))

tissue_type 1885 / 2000 [('left ventricle', 222), ('whole blood', 132), ('skeletal muscle', 131), ('heart', 111)]
cell_type 241 / 2000 [('mucosal surface', 38), ('t-cell', 28), ('fibroblast', 25), ('luteal cells', 20)]
dev_stage 372 / 2000 [('adult', 156), ('fetus', 48), ('terminal', 32), ('kitten', 25)]
sex 667 / 2000 [('female', 351), ('male', 291), ('pooled male and female', 25)]


## 4. Number of files per run

Most runs have 1 file (single-end) or 2 files (paired-end). Some paired-end runs have a third file with the reads that lost their pair.
The 2,000 runs come from a few dozen studies only: runs of the same study are often replicates.

In [5]:
print("files per run:", Counter(len(run["fastq"]) for run in cat["runs"]))
print("studies:", len({run["study_accession"] for run in cat["runs"]}))

files per run: Counter({2: 1295, 1: 608, 3: 97})
studies: 93


## 5. A species with too many runs (human)

Human has millions of RNA-seq runs, so the script never asks for all of them. Here the limit is set to 5 to keep the example short.
`complete` is `False`: we only got part of the runs, and they are the first ones ENA sends, not a representative sample.

In [6]:
human = show(9606, max_runs=5)
[run["run_accession"] for run in human["runs"]]

taxid: 9606 | data type: RNA-Seq
runs in ENA: 2901504 | runs returned: 5 | complete: False


['DRR001173', 'DRR001622', 'DRR001626', 'DRR001628', 'DRR001629']

## 6. Other data types

The same function works for other data types (the name must be written exactly like in ENA: `RNA-Seq`, `ChIP-Seq`, `Hi-C`...).
The Egyptian fruit bat (`9407`) is one of the bats of the test cases.

In [7]:
show(9685, "ChIP-Seq")
show(9685, "Hi-C")
bat = show(9407)

taxid: 9685 | data type: ChIP-Seq
runs in ENA: 230 | runs returned: 230 | complete: True


taxid: 9685 | data type: Hi-C
runs in ENA: 14 | runs returned: 14 | complete: True


taxid: 9407 | data type: RNA-Seq
runs in ENA: 361 | runs returned: 361 | complete: True


## 7. Problems

Each case gives a readable message instead of crashing, or just 0 runs when ENA has nothing.
A taxid that does not exist gives 0 runs.
ENA itself doesn't refuse a data type with a typo (`RNAseq`), it would find 0 runs, so the script checks the data type against ENA's list.
Upper and lower case don't matter (`rna-seq` becomes `RNA-Seq`).

In [8]:
show("abc")
show(9999999)
show(9685, "RNAseq")
show(9407, "rna-seq");

PROBLEM: "abc" is not a valid taxid (it must be a number, e.g. 9685)
taxid: 9999999 | data type: RNA-Seq
runs in ENA: 0 | runs returned: 0 | complete: True
PROBLEM: "RNAseq" is not a data type known by ENA (e.g. RNA-Seq, ChIP-Seq, Hi-C, WGS)


taxid: 9407 | data type: RNA-Seq
runs in ENA: 361 | runs returned: 361 | complete: True
